In [ ]:
import itertools as it

from IPython.display import display
import marimo as mo
from matplotlib import ticker as mpl_ticker
import pandas as pd
import polars as pl
import seaborn as sns
from teeplot import teeplot as tp
from watermark import watermark

from pylib._osf_fetch import osf_fetch
from pylib.munge._calc_fixprobs_from_traits import (
    calc_fixprobs_from_traits,
)
from pylib.theory._calc_denovo_fixprob import calc_denovo_fixprob
from pylib.theory._calc_denovo_fixprob_demes2d import (
    calc_denovo_fixprob_demes2d,
)
from pylib.theory._calc_mutator_fixprob_closed import (
    calc_mutator_fixprob_closed,
)
from pylib.viz._add_legend_entry import add_legend_entry

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-25T22:22:59.120650+00:00

Python implementation: CPython
Python version       : 3.10.22
IPython version      : 7.31.1

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 7.0.0-1017-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

pandas    : 2.2.3
teeplot   : 1.2.0
matplotlib: 3.9.2
seaborn   : 0.13.2
polars    : 1.9.0
marimo    : 0.23.2

```

In [ ]:
teeplot_subdir = "cupy-traits"
teeplot_subdir

'cupy-traits'

## Prep Data

In [ ]:
dfxs = pl.concat(
    [
        pl.from_pandas(
            calc_fixprobs_from_traits(
                pd.read_parquet(
                    osf_fetch("https://osf.io/s67d2/download")
                ),
            ),
        ).with_columns(
            pl.lit("50/50").alias("initial conditions"),
            pl.lit("1D demes").alias("population structure"),
        ),
        pl.from_pandas(
            calc_fixprobs_from_traits(
                pd.read_parquet(
                    osf_fetch("https://osf.io/8q5v6/download")
                ),
            ),
        ).with_columns(
            pl.lit("50/50").alias("initial conditions"),
            pl.lit("2D demes").alias("population structure"),
        ),
        pl.from_pandas(
            calc_fixprobs_from_traits(
                pd.read_parquet(
                    osf_fetch("https://osf.io/duam2/download")
                ),
            ),
        ).with_columns(
            pl.lit("50/50").alias("initial conditions"),
            pl.lit("well-mixed").alias("population structure"),
        ),
        pl.from_pandas(
            calc_fixprobs_from_traits(
                pd.read_parquet(
                    osf_fetch("https://osf.io/seuyf/download")
                ),
            ),
        ).with_columns(
            pl.lit("de novo").alias("initial conditions"),
            pl.lit("1D demes").alias("population structure"),
        ),
        pl.from_pandas(
            calc_fixprobs_from_traits(
                pd.read_parquet(
                    osf_fetch("https://osf.io/ag4ur/download")
                ),
            ),
        ).with_columns(
            pl.lit("de novo").alias("initial conditions"),
            pl.lit("2D demes").alias("population structure"),
        ),
        pl.from_pandas(
            calc_fixprobs_from_traits(
                pd.read_parquet(
                    osf_fetch("https://osf.io/edwbu/download")
                ),
            ),
        ).with_columns(
            pl.lit("de novo").alias("initial conditions"),
            pl.lit("well-mixed").alias("population structure"),
        ),
    ],
)

display(dfxs.describe()), display(dfxs.head()), display(dfxs.tail())

downloading https://osf.io/s67d2/download -> /tmp/s67d2
size: 30888588 bytes


100%|██████████| 56/56 [00:00<00:00, 447.15it/s]


downloading https://osf.io/8q5v6/download -> /tmp/8q5v6
size: 34632554 bytes


100%|██████████| 70/70 [00:00<00:00, 445.01it/s]


downloading https://osf.io/duam2/download -> /tmp/duam2
size: 25587641 bytes


100%|██████████| 56/56 [00:00<00:00, 477.82it/s]


... (2 outputs left out)

downloading https://osf.io/ag4ur/download -> /tmp/ag4ur
size: 85511876 bytes


100%|██████████| 154/154 [00:00<00:00, 416.66it/s]


downloading https://osf.io/edwbu/download -> /tmp/edwbu
size: 59971615 bytes


100%|██████████| 126/126 [00:00<00:00, 460.11it/s]
sys:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance


## Size Fixation Cliffplot

In [ ]:
# math model predictions (see pylib/theory), for well-mixed populations
# and for a 2D lattice of 256-agent demes
model_grid = [1, 2, 3, 4, 6, 8, 10, 12, 14, 16, 20, 24, 32, 40]
data_K = (
    dfxs.filter(pl.col("population structure") != "1D demes")
    .to_pandas()
    .groupby(
        ["initial conditions", "population structure", "population size"]
    )["available beneficial mutations"]
    .agg(["min", "max"])
)
# predictions extend only as far along the x axis as the data do
model_xy = {}
for (_init, _structure, _N), _range in data_K.iterrows():
    _calc = {
        ("50/50", "well-mixed"): calc_mutator_fixprob_closed,
        ("50/50", "2D demes"): lambda N, K: calc_mutator_fixprob_closed(
            N, K, deme_size=256
        ),
        ("de novo", "well-mixed"): calc_denovo_fixprob,
        ("de novo", "2D demes"): calc_denovo_fixprob_demes2d,
    }[_init, _structure]
    _Ks = [K for K in model_grid if _range["min"] <= K <= _range["max"]]
    model_xy.setdefault(_N, []).append(
        (_Ks, [_calc(int(_N), K) for K in _Ks]),
    )

In [ ]:
data = dfxs.filter(
    pl.col("genotype") == "hypermutator",
    ~(
        (pl.col("population size") > 1679616)
        & (pl.col("population structure") == "1D demes")
    ),
    ~(
        (pl.col("population size") <= 256)
        & (pl.col("available beneficial mutations") > 12)
        & (pl.col("population structure") == "1D demes")
    ),
).with_columns(
    pl.col("fixation probability").alias("fix\nprob"),
    pl.col("population structure").alias("population\nstructure"),
    pl.col("initial conditions").alias("initial\nconditions"),
)

row = "population size"
for errorbar, exclude in it.product(
    ["sd", "se", "ci", None],
    ["1D demes", None],
):
    with tp.teed(
        sns.relplot,
        data=data.filter(
            pl.col("population structure") != pl.lit(exclude)
            if exclude is not None
            else True,
        ),
        x="available beneficial mutations",
        y="fix\nprob",
        row=row,
        row_order=sorted(data[row].to_pandas().unique(), reverse=True),
        hue="population\nstructure",
        hue_order=[
            *filter(
                lambda x: x != exclude,
                ("well-mixed", "2D demes", "1D demes"),
            ),
        ],
        style="initial\nconditions",
        style_order=["de novo", "50/50"],
        aspect=10,
        errorbar=errorbar,
        # facet_kws=dict(margin_titles=True),
        height=0.8,
        kind="line",
        markers=True,
        palette="Dark2",
        seed=1,
        teeplot_subdir=teeplot_subdir,
        teeplot_outattrs=(
            {"exclude": exclude.replace(" ", "-")}
            if exclude is not None
            else {}
        ),
    ) as teed:
        for _model_N, _model_ax in teed.axes_dict.items():
            for _model_curve in model_xy.get(_model_N, []):
                _model_ax.plot(
                    *_model_curve, color="black", linestyle="--"
                )
        teed.set_titles(
            col_template="",
            row_template="",
        )
        teed.set(ylim=(0, 1), xlim=(1, 40))
        sns.move_legend(
            teed,
            "lower center",
            bbox_to_anchor=(0.4, 0.97),
            frameon=False,
            ncol=7,
            title=None,
            columnspacing=0.7,
        )
        add_legend_entry(teed, "analytic\napprox")
        for ax in teed.axes.flat:
            ax.axhline(0.5, ls=":", c="gray", lw=1)
            ax.set_yticks([0, 0.5, 1])
            ax.set_yticklabels(["", "0.5", ""])
            ax.set_ylabel("")

        teed.tight_layout()

        teed.figure.subplots_adjust(top=0.85)
        delta = 0.001  # Small height for the dummy axis
        pos = teed.axes.flat[0].get_position()
        dummy_ax = ax.figure.add_axes(
            [
                pos.x0 + pos.width * 1.05,
                pos.y0
                - pos.height * data[row].to_pandas().nunique() * 1.16,
                delta,
                pos.height * data[row].to_pandas().nunique() * 1.58,
            ],
        )
        yvals = data[row]
        dummy_ax.set_ylim(yvals.min(), yvals.max())
        dummy_ax.yaxis.set_label_position("right")
        dummy_ax.set_ylabel(row)

        dummy_ax.set_yscale("log")
        dummy_ax.yaxis.set_ticks_position("right")
        dummy_ax.yaxis.set_major_locator(
            mpl_ticker.LogLocator(base=10),
        )
        formatter = mpl_ticker.LogFormatterMathtext(base=10)
        dummy_ax.yaxis.set_major_formatter(formatter)

        # Hide unwanted spines
        dummy_ax.get_xaxis().set_visible(False)

        teed.tight_layout()

/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/errorbar=sd+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.pdf
teeplots/cupy-traits/errorbar=sd+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.png


/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/errorbar=sd+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.pdf
teeplots/cupy-traits/errorbar=sd+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.png


/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/errorbar=se+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.pdf
teeplots/cupy-traits/errorbar=se+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.png


... (6 outputs left out)

/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.pdf
teeplots/cupy-traits/exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.png


/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.pdf
teeplots/cupy-traits/hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+y=fix-prob+ext=.png


### Log-scaled x axis

In [ ]:
_data = dfxs.filter(
    pl.col("genotype") == "hypermutator",
    ~(
        (pl.col("population size") > 1679616)
        & (pl.col("population structure") == "1D demes")
    ),
    ~(
        (pl.col("population size") <= 256)
        & (pl.col("available beneficial mutations") > 12)
        & (pl.col("population structure") == "1D demes")
    ),
).with_columns(
    pl.col("fixation probability").alias("fix\nprob"),
    pl.col("population structure").alias("population\nstructure"),
    pl.col("initial conditions").alias("initial\nconditions"),
)

_row = "population size"
for _errorbar, _exclude in it.product(
    ["sd", "se", "ci", None],
    ["1D demes", None],
):
    with tp.teed(
        sns.relplot,
        data=_data.filter(
            pl.col("population structure") != pl.lit(_exclude)
            if _exclude is not None
            else True,
        ),
        x="available beneficial mutations",
        y="fix\nprob",
        row=_row,
        row_order=sorted(_data[_row].to_pandas().unique(), reverse=True),
        hue="population\nstructure",
        hue_order=[
            *filter(
                lambda x: x != _exclude,
                ("well-mixed", "2D demes", "1D demes"),
            ),
        ],
        style="initial\nconditions",
        style_order=["de novo", "50/50"],
        aspect=10,
        errorbar=_errorbar,
        # facet_kws=dict(margin_titles=True),
        height=0.8,
        kind="line",
        markers=True,
        palette="Dark2",
        seed=1,
        teeplot_subdir=teeplot_subdir,
        teeplot_outattrs={
            "xscale": "log",
            **(
                {"exclude": _exclude.replace(" ", "-")}
                if _exclude is not None
                else {}
            ),
        },
    ) as _teed:
        for _model_N, _model_ax in _teed.axes_dict.items():
            for _model_curve in model_xy.get(_model_N, []):
                _model_ax.plot(
                    *_model_curve, color="black", linestyle="--"
                )
        _teed.set_titles(
            col_template="",
            row_template="",
        )
        _teed.set(ylim=(0, 1), xlim=(1, 40))
        sns.move_legend(
            _teed,
            "lower center",
            bbox_to_anchor=(0.4, 0.97),
            frameon=False,
            ncol=7,
            title=None,
            columnspacing=0.7,
        )
        add_legend_entry(_teed, "analytic\napprox")
        for _ax in _teed.axes.flat:
            _ax.axhline(0.5, ls=":", c="gray", lw=1)
            _ax.set_yticks([0, 0.5, 1])
            _ax.set_yticklabels(["", "0.5", ""])
            _ax.set_ylabel("")

        _teed.tight_layout()

        _teed.figure.subplots_adjust(top=0.85)
        _teed.set(xscale="log")
        for _ax in _teed.axes.flat:
            _ax.set_xticks([1, 2, 4, 8, 16, 40])
            _ax.xaxis.set_major_formatter(mpl_ticker.ScalarFormatter())
            _ax.xaxis.set_minor_formatter(mpl_ticker.NullFormatter())
        _delta = 0.001  # Small height for the dummy axis
        _pos = _teed.axes.flat[0].get_position()
        _dummy_ax = _ax.figure.add_axes(
            [
                _pos.x0 + _pos.width * 1.05,
                _pos.y0
                - _pos.height * _data[_row].to_pandas().nunique() * 1.16,
                _delta,
                _pos.height * _data[_row].to_pandas().nunique() * 1.58,
            ],
        )
        _yvals = _data[_row]
        _dummy_ax.set_ylim(_yvals.min(), _yvals.max())
        _dummy_ax.yaxis.set_label_position("right")
        _dummy_ax.set_ylabel(_row)

        _dummy_ax.set_yscale("log")
        _dummy_ax.yaxis.set_ticks_position("right")
        _dummy_ax.yaxis.set_major_locator(
            mpl_ticker.LogLocator(base=10),
        )
        _formatter = mpl_ticker.LogFormatterMathtext(base=10)
        _dummy_ax.yaxis.set_major_formatter(_formatter)

        # Hide unwanted spines
        _dummy_ax.get_xaxis().set_visible(False)

        _teed.tight_layout()

/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/errorbar=sd+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.pdf
teeplots/cupy-traits/errorbar=sd+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.png


/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/errorbar=sd+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.pdf
teeplots/cupy-traits/errorbar=sd+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.png


/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/errorbar=se+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.pdf
teeplots/cupy-traits/errorbar=se+exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.png


... (6 outputs left out)

/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.pdf
teeplots/cupy-traits/exclude=1D-demes+hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.png


/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  self._figure.tight_layout(*args, **kwargs)


teeplots/cupy-traits/hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.pdf
teeplots/cupy-traits/hue=population-structure+kind=line+palette=dark2+row=population-size+style=initial-conditions+viz=relplot+x=available-beneficial-mutations+xscale=log+y=fix-prob+ext=.png
